# 多流表达功能

## 1. 功能简介

在大模型推理中，如果计算分支之间不存在数据依赖，可以将任务分发到不同 Stream，使计算或通信在时间线上重叠（overlap），从而缩短端到端耗时。多流优化的前提是依赖关系正确、设备资源仍有余量，并且并行收益高于额外的调度与同步开销。

常见并行方式包括：

- **计算—计算并行**：识别互不依赖的计算分支，并将其分配到不同 Stream。
- **计算—通信并行**：在依赖关系允许时，让通信任务与计算任务重叠执行。

该能力主要用于 **aclgraph 之间的资源并发**，更适合 Cube 计算资源尚未饱和的场景。如果单流已占满计算资源，多流可能只会增加调度与同步成本。

> **多流不是越多越快**。如果 Cube 资源已经饱和，额外调度可能导致性能下降。

## 2. 使用约束

- 本功能支持的产品型号参见[使用说明](https://gitcode.com/Ascend/torchair/blob/master/docs/zh/overview.md#%E4%BD%BF%E7%94%A8%E8%AF%B4%E6%98%8E)。
- 同一个 event 只能在一张整图内使用，不能跨越 graph break（即需在同一编译段内）。

## 3. 使用方法

1. **分析可并行算子**：用户自行分析模型脚本中可进行并行计算的算子。
2. **开启多流表达**：使用 `with` 语句块，语句块内下发的算子切换至 `stream` 参数指定的流计算，语句块外的算子使用默认流计算。
   ```python
   with torch.npu.stream(stream: torch.npu.Stream):
   ```
3. **控制并行计算时序（可选）**：通过 `torch.npu.Event()`、`torch.npu.Event.record()`、`torch.npu.Event.wait()` 系列原生接口实现时序控制。
4. **延长内存释放时机（可选）**：Eager 模式场景下，脚本中如果涉及多 stream 内存复用，一般会调用 PyTorch 的 `tensor.record_stream` 原生接口延迟内存释放。
5. **配置限核（可选）**：参考 [AI Core 和 Vector Core 限核功能](https://gitcode.com/Ascend/torchair/blob/master/docs/zh/npugraph%5Fex/advanced/limit%5Fcores.md)，避免单条 Stream 独占所有计算核，影响并行效果。

## 4. 使用示例

下面的示例展示了如何在 `npugraph_ex` 后端中使用多流表达。模型中包含 4 个输入张量，通过创建两条额外 stream（`stream1`、`stream2`）和一个默认流，将部分计算任务分发到不同流上并行执行，同时使用 event 控制流间数据依赖。

In [ ]:
import math

import torch
import torch_npu


DEVICE = "npu"
DTYPE = torch.float16

# 受控的微批处理示例：
# 两条相互独立的 GEMM 分支使用相同的矩阵形状和计算深度，
# 因此两条分支的执行时间天然比较接近。
MATRIX_SIZE = 1024
CUBE_DEPTH = 8

# 假设设备有 24 个 AI/Cube Core 和 48 个 Vector Core。
# 在单流和多流版本中，每条分支都分配一半的计算资源：
#   单流：分支 A -> 分支 B
#   多流：分支 A || 分支 B
#
# 不同 NPU 的核数可能不同，请从 CANN 的 platform_config/*.ini 文件中
# 读取实际核数，并相应调整下面两个参数。配置值不能超过设备上限。
AICORE_PER_BRANCH = 12
VECTORCORE_PER_BRANCH = 24


def gemm_branch(x, weight, depth):
    """执行一条由 GEMM 和 ReLU 组成的计算分支。"""
    for _ in range(depth):
        x = torch.mm(x, weight)
        x = torch.relu(x)
    return x


def limit_current_stream():
    # torch_npu 2.7.x 只接受两个核数参数。
    # 该作用域会应用到进入 with 语句时的当前 Stream。
    return torch.npu.npugraph_ex.scope.limit_core_num(
        AICORE_PER_BRANCH,
        VECTORCORE_PER_BRANCH,
    )


class SingleStreamModel(torch.nn.Module):
    """单流基线：执行相同的两条受限分支，但在同一条流上串行执行。"""

    def forward(self, x_a, weight_a, x_b, weight_b):
        with limit_current_stream():
            result_a = gemm_branch(x_a, weight_a, CUBE_DEPTH)
        with limit_current_stream():
            result_b = gemm_branch(x_b, weight_b, CUBE_DEPTH)
        return result_a, result_b


class MultiStreamModel(torch.nn.Module):
    """多流版本：将两条分支分别提交到不同的 Stream 上执行。"""

    def __init__(self):
        super().__init__()
        self.stream_a = torch.npu.Stream()
        self.stream_b = torch.npu.Stream()

    def forward(self, x_a, weight_a, x_b, weight_b):
        entry_stream = torch.npu.current_stream()
        ready = torch.npu.Event()
        done_a = torch.npu.Event()
        done_b = torch.npu.Event()

        # 让两个从流等待入口流之前已经提交的任务。
        # 两条分支之间没有数据依赖，因此可以并行执行。
        ready.record()

        with torch.npu.stream(self.stream_a):
            ready.wait(self.stream_a)
            with torch.npu.npugraph_ex.scope.limit_core_num(
                AICORE_PER_BRANCH,
                VECTORCORE_PER_BRANCH,
            ):
                result_a = gemm_branch(x_a, weight_a, CUBE_DEPTH)
            done_a.record()

        with torch.npu.stream(self.stream_b):
            ready.wait(self.stream_b)
            with torch.npu.npugraph_ex.scope.limit_core_num(
                AICORE_PER_BRANCH,
                VECTORCORE_PER_BRANCH,
            ):
                result_b = gemm_branch(x_b, weight_b, CUBE_DEPTH)
            done_b.record()

        # 只有两个从流都执行完成后才返回。
        # 这两个等待也属于本次基准测试的关键路径。
        done_a.wait(entry_stream)
        done_b.wait(entry_stream)
        return result_a, result_b


def describe(name, tensor):
    tensor_float = tensor.float()
    print(
        f"  {name:10s} | shape={tuple(tensor.shape)} "
        f"| dtype={tensor.dtype} "
        f"| mean={tensor_float.mean().item():+.4f} "
        f"| max={tensor_float.max().item():+.4f} "
        f"| min={tensor_float.min().item():+.4f}"
    )


def benchmark(fn, args, warmup=10, iters=50):
    """使用 NPU Event 统计设备侧平均执行时间。"""
    for _ in range(warmup):
        fn(*args)
    torch.npu.synchronize()

    start = torch.npu.Event(enable_timing=True)
    end = torch.npu.Event(enable_timing=True)
    start.record()
    for _ in range(iters):
        fn(*args)

    # 记录结束时间前，先等待所有从流任务完成，确保不会漏测多流任务。
    torch.npu.synchronize()
    end.record()
    torch.npu.synchronize()
    return start.elapsed_time(end) / iters


def max_abs_diff(lhs, rhs):
    return (lhs.float() - rhs.float()).abs().max().item()


def main():
    torch.npu.set_device(0)

    size = MATRIX_SIZE
    x_a = torch.randn(size, size, dtype=DTYPE, device=DEVICE) * 0.1
    weight_a = torch.randn(size, size, dtype=DTYPE, device=DEVICE) / math.sqrt(size)
    x_b = torch.randn(size, size, dtype=DTYPE, device=DEVICE) * 0.1
    weight_b = torch.randn(size, size, dtype=DTYPE, device=DEVICE) / math.sqrt(size)
    args = (x_a, weight_a, x_b, weight_b)

    single = SingleStreamModel().to(DEVICE)
    multi = MultiStreamModel().to(DEVICE)

    # 两个版本使用相同的 npugraph_ex 后端和编译选项，
    # 避免把编译方式差异误认为是多流带来的收益。
    single_compiled = torch.compile(
        single,
        backend="npugraph_ex",
        fullgraph=True,
        dynamic=False,
    )
    multi_compiled = torch.compile(
        multi,
        backend="npugraph_ex",
        fullgraph=True,
        dynamic=False,
    )

    print(
        f"workload: size={size}, depth={CUBE_DEPTH}, "
        f"cores_per_branch={AICORE_PER_BRANCH} AI/"
        f"{VECTORCORE_PER_BRANCH} Vector"
    )

    print("\n[1] Multi-stream output:")
    multi_result = multi_compiled(*args)
    torch.npu.synchronize()
    for name, tensor in zip(("result_a", "result_b"), multi_result):
        describe(name, tensor)

    print("\n[2] Correctness check:")
    single_result = single_compiled(*args)
    torch.npu.synchronize()
    for name, got, expected in zip(
        ("result_a", "result_b"), multi_result, single_result
    ):
        diff = max_abs_diff(got, expected)
        status = "PASS" if diff < 1e-2 else "FAIL"
        print(f"  {name:10s} | max_abs_diff={diff:.6e} | {status}")

    print("\n[3] Performance:")
    single_ms = benchmark(single_compiled, args)
    multi_ms = benchmark(multi_compiled, args)
    speedup = single_ms / multi_ms
    print(f"  single stream : {single_ms:.3f} ms / iter")
    print(f"  multi stream  : {multi_ms:.3f} ms / iter")
    print(f"  speedup       : {speedup:.2f}x")
    print("  两条等时长分支的理论上限：约 2.00x")

    if speedup <= 1.0:
        print(
            "\nNo speedup was observed. Check the following first:"
            "\n  1. The per-branch core limits are valid for this NPU."
            "\n  2. The profiler shows both side streams overlapping."
            "\n  3. The installed CANN/torch_npu version supports core limits"
            " for the selected GEMM operator."
        )


if __name__ == "__main__":
    main()


## 5. 执行流程说明

*图1 多流表达示意图*

![多流表达示意图](./images/multi_stream_representation.png)

多流表达示意图中实线框展示了多流并行执行的时序控制关系，用户在脚本中通过 `event1` 和 `event2` 控制流间数据依赖，确保具有依赖关系的计算任务按正确顺序执行。虚线框标注了 `npugraph_ex` 在编图时自动插入的 `record`、`wait` 等同步点，形成执行闭环，目的是让从流的算子能够被捕获到（CUDA Graph 相同逻辑，切换到从流需要用开头和结尾的两对 `record`、`wait` 以实现流捕获）。

### 5.1 时序关系

1. **默认流**：先执行 `add_result = in1 + in2` 和 `branch = in3 + in4`，然后 `event1.record()` 记录进度。
2. **stream1**：`event1.wait(stream1)` 等待默认流完成，然后执行 `mm_result = branch @ in4`，完成后 `event2.record()` 记录进度。
3. **默认流**（可与 stream1 并行）：执行 `mm_default = in3 @ in4`。
4. **stream2**：`event2.wait(stream2)` 等待 stream1 完成，然后执行 `add_after_mm = in3 + in4`。

### 5.2 关键接口说明

<table align="left" border="1" cellpadding="6" cellspacing="0">
  <tr><th align="left">接口</th><th align="left">作用</th></tr>
  <tr><td align="left"><code>torch.npu.Stream()</code></td><td align="left">创建一条新的 NPU 流。</td></tr>
  <tr><td align="left"><code>torch.npu.stream(stream)</code></td><td align="left">上下文管理器，块内算子下发到指定 Stream。</td></tr>
  <tr><td align="left"><code>torch.npu.Event()</code></td><td align="left">创建事件，用于流间同步。</td></tr>
  <tr><td align="left"><code>event.record()</code></td><td align="left">在当前流上记录事件。</td></tr>
  <tr><td align="left"><code>event.wait(stream)</code></td><td align="left">指定 Stream 等待该事件完成。</td></tr>
  <tr><td align="left"><code>tensor.record_stream(stream)</code></td><td align="left">延长 Tensor 的内存生命周期，防止其仍在 Stream 上使用时被回收。</td></tr>
</table>
<div style="clear: both;"></div>

## 6. 关键约束与注意事项

- 同一个 event 只能在一张整图内使用，不能跨 graph break（即需在同一编译段内）。
- 有依赖关系时必须使用 `Event.record` 和 `Event.wait`，否则可能出现数据竞争。
- 在非默认流使用 Tensor 时，必要时用 `record_stream` 延长内存释放，防止内存提前回收。
- 可以与限核功能组合使用，防止一个流占满所有核导致其他流饥饿。
- 必须用 Profiler 时间线验证多流是否真正形成 overlap，是否有实际收益。

## 7. 课后练习

### 一、单选题

（1）【单选题】将语句块内算子下发到指定 NPU 流的上下文管理器是？
- A. `with torch.npu.stream(stream):`
- B. `with torch.compile(stream):`
- C. `with torch.npu.Event():`
- D. `with torch.npu.synchronize():`

（2）【单选题】多流之间存在数据依赖时，推荐使用什么机制控制执行顺序？
- A. 修改 Tensor dtype
- B. `Event.record()` 与 `Event.wait()`
- C. 关闭图模式
- D. 每次重启进程

（3）【单选题】在非默认流仍使用某个 Tensor 时，`tensor.record_stream(stream)` 的主要作用是？
- A. 将 Tensor 复制到 CPU
- B. 延长 Tensor 的内存生命周期，避免被提前回收
- C. 强制所有流同步
- D. 增加流数量

（4）【单选题】下列哪种场景最可能从多流中获益？
- A. 单流已经完全占满 Cube 计算资源
- B. 存在互不依赖的计算分支且设备资源有余量
- C. 所有算子必须严格串行执行
- D. 不希望进行 Profiling

（5）【单选题】同一个 NPU Event 的使用约束是？
- A. 可以跨任意 graph break 使用
- B. 只能在同一张整图内使用，不能跨 graph break
- C. 只能用于 CPU Stream
- D. 必须在多个进程共享

（6）【单选题】确认多流是否形成有效 overlap 的推荐方法是？
- A. 仅增加 Stream 数量
- B. 使用 Profiler 查看时间线和端到端耗时
- C. 删除所有 Event
- D. 只查看模型参数量

### 二、多选题

（7）【多选题】关于多流表达，正确的说法有哪些？
- A. 可用于计算—计算并行
- B. 可用于计算—通信并行
- C. 并行收益需要大于额外的调度和同步开销
- D. 流数量越多，性能一定越高

（8）【多选题】关于 Event 的使用，正确的说法有哪些？
- A. `event.record()` 用于记录当前流的执行进度
- B. `event.wait(stream)` 可让指定流等待事件完成
- C. 有依赖关系时应使用 Event 建立同步关系
- D. Event 可以随意跨 graph break 复用

（9）【多选题】多流场景下正确管理 Tensor 生命周期的做法有哪些？
- A. 在非默认流继续使用 Tensor 时，必要时调用 `record_stream`
- B. 确保内存不会在从流使用结束前被回收
- C. 结合 Event 表达真实的数据依赖
- D. 假设默认流会自动处理所有跨流内存问题

（10）【多选题】评估多流优化是否有效，应关注哪些方面？
- A. Profiler 时间线是否出现实际 overlap
- B. 端到端耗时是否改善
- C. 计算资源是否仍有余量并且依赖关系正确
- D. 仅凭创建了多条 Stream 就认定优化成功

**运行以下代码单元查看参考答案与解析。**


In [ ]:
import os
answer_path = "answer/04.03_answer.txt"
if os.path.exists(answer_path):
    with open(answer_path, "r", encoding="utf-8") as f:
        print(f.read())
else:
    print("答案文件未找到，请检查 answer 目录。")
